# San Antonio candidate bicyclist-injury corridors

This notebook analyzes all qualifying San Antonio CRIS crashes from 2024 through Sept. 1, 2026. It does not call or filter against the City's official HIN layer. It identifies repeat-crash stretches using local street segments, then creates a clickable map with victim details.

In [ ]:
from pathlib import Path
import html, zipfile
import pandas as pd
import geopandas as gpd
from sklearn.cluster import AgglomerativeClustering
import folium

ROOT = Path.cwd(); RAW = ROOT/'data'/'raw'; OUT = ROOT/'outputs'
OUT.mkdir(exist_ok=True)

raw = pd.read_csv(RAW/'myexport_final.csv', skiprows=12, low_memory=False)
severity = {'K - FATAL INJURY','A - SUSPECTED SERIOUS INJURY'}
target = raw[(raw['City']=='SAN ANTONIO') & (raw['Person Type']=='3 - PEDALCYCLIST') & raw['Person Injury Severity'].isin(severity)].copy()
target['year'] = pd.to_numeric(target['Crash Year'], errors='coerce')
target = target[target['year'].between(2024,2026)].copy()
target['latitude'] = pd.to_numeric(target['Latitude'], errors='coerce')
target['longitude'] = pd.to_numeric(target['Longitude'], errors='coerce')
target['death'] = (target['Person Injury Severity']=='K - FATAL INJURY').astype(int)
target['serious_injury'] = (target['Person Injury Severity']=='A - SUSPECTED SERIOUS INJURY').astype(int)
factor_cols = ['Contributing Factor 1','Contributing Factor 2','Contributing Factor 3']
target['contributing_factors'] = target[factor_cols].fillna('').astype(str).replace({'nan':'','None':''},regex=False).agg('; '.join,axis=1).str.replace(r'(; )+','; ',regex=True).str.strip('; ')
def combine_values(s): return '; '.join(sorted({str(v).strip() for v in s.dropna() if str(v).strip() and str(v).strip().upper() not in {'NAN','NONE','TBD'}}))
crashes = target.groupby('Crash ID',as_index=False).agg(year=('year','first'),latitude=('latitude','first'),longitude=('longitude','first'),deaths=('death','sum'),serious_injuries=('serious_injury','sum'),victim_ages=('Person Age',combine_values),victim_genders=('Person Gender',combine_values),victim_helmets=('Person Helmet',combine_values),contributing_factors=('contributing_factors',combine_values))
geo = crashes.dropna(subset=['latitude','longitude']).copy()
points = gpd.GeoDataFrame(geo,geometry=gpd.points_from_xy(geo['longitude'],geo['latitude']),crs=4326)
print('Qualifying crashes:',len(points)); print('Deaths:',int(points.deaths.sum())); print('Serious injuries:',int(points.serious_injuries.sum()))

In [ ]:
# Match each crash to the nearest local street segment.
streets_dir = RAW/'streets'; street_shp = streets_dir/'Streets'/'Streets.shp'
if not street_shp.exists():
    with zipfile.ZipFile(RAW/'Streets.zip') as z: z.extractall(streets_dir)
roads = gpd.read_file(street_shp).rename(columns={'CartID':'segmentid','MSAG_NAME':'road_label','FROM_STREE':'from_street','TO_STREET':'to_street','CoSARoadFu':'road_class'})
roads['length_miles'] = pd.to_numeric(roads['LengthFeet'],errors='coerce')/5280
roads = roads[roads['length_miles']>0].copy().to_crs(2279)
fields = ['segmentid','road_label','from_street','to_street','road_class','length_miles','geometry']
matches = gpd.sjoin_nearest(points.to_crs(roads.crs),roads[fields],how='left',distance_col='match_distance_ft')
matches = matches[matches['match_distance_ft']<=150].sort_values('match_distance_ft').drop_duplicates('Crash ID').copy()

# The original City HIN corridors topped out at about three miles. Use that as a transparent maximum stretch length.
MAX_CORRIDOR_MILES = 3.0
matches['candidate_group'] = pd.NA; group_number = 0
for road_name,group in matches.groupby('road_label',dropna=False):
    if len(group)<2 or pd.isna(road_name) or not str(road_name).strip(): continue
    coords=[[p.x,p.y] for p in group.geometry]
    labels=AgglomerativeClustering(n_clusters=None,distance_threshold=MAX_CORRIDOR_MILES*5280,linkage='complete').fit_predict(coords)
    for label in sorted(set(labels)):
        idx=group.index[labels==label]
        if len(idx)>=2: matches.loc[idx,'candidate_group']=group_number; group_number+=1
matches=matches[matches['candidate_group'].notna()].copy()
def join_values(s): return ', '.join(sorted({str(v).strip() for v in s.dropna() if str(v).strip() and str(v).strip().upper() not in {'NAN','NONE','TBD'}}))
corridors=matches.groupby('candidate_group',as_index=False).agg(crashes=('Crash ID','nunique'),deaths=('deaths','sum'),serious_injuries=('serious_injuries','sum'),first_year=('year','min'),last_year=('year','max'),roads=('road_label',join_values),from_streets=('from_street',join_values),to_streets=('to_street',join_values),max_match_distance_ft=('match_distance_ft','max'))
spans=[]
for gid,g in matches.groupby('candidate_group'): spans.append({'candidate_group':gid,'span_miles':max(g.geometry.x.max()-g.geometry.x.min(),g.geometry.y.max()-g.geometry.y.min())/5280})
corridors=corridors.merge(pd.DataFrame(spans),on='candidate_group',how='left').sort_values(['crashes','deaths','serious_injuries'],ascending=False)
corridors.to_csv(OUT/'candidate_corridors_2024_2026.csv',index=False)
print('Candidate repeat-crash corridors:',len(corridors)); display(corridors)

In [ ]:
# Clickable map of corridors and every qualifying crash.
segment_groups=matches[['segmentid','candidate_group']].dropna().drop_duplicates()
candidate_lines=roads.merge(segment_groups,on='segmentid',how='inner').dissolve(by='candidate_group',as_index=False).merge(corridors,on='candidate_group',how='left')
def shown(value):
    text='' if pd.isna(value) else str(value).strip(); return html.escape(text) if text else 'Not recorded'
m=folium.Map(location=[points.geometry.y.mean(),points.geometry.x.mean()],zoom_start=11,tiles='OpenStreetMap',control_scale=True)
folium.GeoJson(candidate_lines.to_crs(4326).to_json(),name='Candidate corridors',style_function=lambda f:{'color':'#d95f02','weight':6,'opacity':.85},tooltip=folium.GeoJsonTooltip(fields=['roads','crashes','deaths','serious_injuries','first_year','last_year'],aliases=['Road','Crashes','Deaths','Serious injuries','First year','Last year'])).add_to(m)
for _,row in points.iterrows():
    outcome=f"{int(row.deaths)} death(s), {int(row.serious_injuries)} serious injury/ies"
    popup=(f"<b>Crash {shown(row['Crash ID'])}</b><br>Year: {int(row.year)}<br>Outcome: {outcome}<br>"
           f"Victim age: {shown(row.victim_ages)}<br>Victim gender: {shown(row.victim_genders)}<br>Helmet: {shown(row.victim_helmets)}<br>"
           f"Recorded contributing factors: {shown(row.contributing_factors)}<br>Latitude: {row.latitude:.6f}<br>Longitude: {row.longitude:.6f}")
    folium.CircleMarker([row.latitude,row.longitude],radius=5,color='#e67e22' if row.deaths else '#3478a4',fill=True,fill_opacity=.9,tooltip=f"{int(row.year)} — Crash {row['Crash ID']}",popup=folium.Popup(popup,max_width=360)).add_to(m)
folium.LayerControl().add_to(m)
map_path=OUT/'candidate_corridors_victim_map_2024_2026.html'; m.save(map_path); print('Map saved to:',map_path)

In [ ]:
# Final standalone Leaflet map. No API key is required.
import json

candidate_groups = matches[['Crash ID','candidate_group']].drop_duplicates('Crash ID')
candidate_points = points.merge(candidate_groups, on='Crash ID', how='inner')
line_geo = candidate_lines.to_crs(4326)
features = []
for _, row in candidate_points.iterrows():
    features.append({'type':'Feature','properties':{'kind':'crash','group':int(row['candidate_group']),'crash':str(row['Crash ID']),'year':int(row['year']),'deaths':int(row['deaths']),'injuries':int(row['serious_injuries']),'ages':shown(row.get('victim_ages')),'genders':shown(row.get('victim_genders')),'helmets':shown(row.get('victim_helmets'))},'geometry':{'type':'Point','coordinates':[float(row.geometry.x),float(row.geometry.y)]}})
for _, row in line_geo.iterrows():
    props = row.drop(labels='geometry').to_dict(); props['kind']='line'; props['group']=int(row['candidate_group'])
    for key, value in list(props.items()):
        if pd.isna(value): props[key]=''
    geometry = json.loads(gpd.GeoSeries([row.geometry], crs=4326).to_json())['features'][0]['geometry']
    features.append({'type':'Feature','properties':props,'geometry':geometry})
geojson_text = json.dumps({'type':'FeatureCollection','features':features}, default=str)
colors = ['#d73027','#4575b4','#1a9850','#984ea3','#f46d43','#66bd63','#3288bd','#762a83','#e66101']
page = f'''<!doctype html><html><head><meta charset="utf-8"><title>San Antonio bicyclist crash clusters, 2024–2026</title><meta name="viewport" content="width=device-width,initial-scale=1"><link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css"><style>body{{margin:0;font:14px Arial}}#map{{height:100vh;background:#e8e6e1}}.legend{{background:white;padding:8px;line-height:18px;box-shadow:0 0 8px #777}}</style></head><body><div id="map"></div><script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script><script>const data={geojson_text};const colors={json.dumps(colors)};const map=L.map('map').setView([29.4241,-98.4936],11);const carto=L.tileLayer('https://{{s}}.basemaps.cartocdn.com/rastertiles/voyager/{{z}}/{{x}}/{{y}}{{r}}.png',{{maxZoom:19,subdomains:'abcd',attribution:'&copy; OpenStreetMap contributors &copy; CARTO'}});const osm=L.tileLayer('https://tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png',{{maxZoom:19,attribution:'&copy; OpenStreetMap contributors'}});let errors=0;carto.on('tileerror',()=>{{if(++errors>4&&!map.hasLayer(osm)){{map.removeLayer(carto);osm.addTo(map)}}}});carto.addTo(map);const lines=L.layerGroup(),pointsLayer=L.layerGroup(),bounds=[];data.features.forEach(f=>{{const p=f.properties;if(p.kind==='line'){{const layer=L.geoJSON(f,{{style:{{color:colors[p.group%colors.length],weight:6,opacity:.85}}}}).bindTooltip('Candidate '+(p.group+1)+' — '+p.roads);layer.addTo(lines);}}else{{const ll=[f.geometry.coordinates[1],f.geometry.coordinates[0]];L.circleMarker(ll,{{radius:6,color:p.deaths?'#111':'#fff',weight:2,fillColor:p.deaths?'#e67e22':'#3478a4',fillOpacity:.95}}).bindPopup('<b>Crash '+p.crash+'</b><br>Year: '+p.year+'<br>Outcome: '+p.deaths+' death(s), '+p.injuries+' serious injury/ies<br>Age: '+p.ages+'<br>Gender: '+p.genders+'<br>Helmet: '+p.helmets).addTo(pointsLayer);bounds.push(ll);}}}});lines.addTo(map);pointsLayer.addTo(map);if(bounds.length)map.fitBounds(bounds,{{padding:[30,30]}});L.control.layers({{'CARTO Voyager':carto,'OpenStreetMap':osm}},{{'Candidate stretches':lines,'Crash locations':pointsLayer}},{{collapsed:false}}).addTo(map);</script></body></html>'''
map_path = OUT/'candidate_corridors_victim_map_2024_2026.html'
map_path.write_text(page)
print('Map saved to:', map_path)
